# Capstone: Define and Solve an ML Problem

In [1]:
import pandas as pd
import numpy as np
import os 
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
import tensorflow.keras as keras
from sklearn.preprocessing import StandardScaler
import time

<class 'ModuleNotFoundError'>: No module named 'seaborn'

**Note**: As you work through the notebook, you can import additional packages as needed.

## Overview


In this capstone assignment, you will follow the machine learning life cycle and implement one of the supervised learning models you have learned so far in this course, along with a neural network, to solve a predictive problem.

This capstone spans two lab sessions.

- **Unit 5 Lab:** You will define a machine learning problem, explore and prepare your data, and train, test, evaluate and improve a traditional machine learning model (Parts 1–5).
- **Unit 6 Lab:** After completing Unit 6 on neural networks, you will apply a neural network to the same problem and compare the two approaches (Parts 6–7).

There is a checkpoint at the end of Part 5 that marks where to stop during the Unit 5 lab.

You will complete the following:

1. Choose your Data Set and Build Your DataFrame
2. Define Your ML Problem
3. Understand Your Data
4. Prepare Your Data
5. Train, Test, Evaluate and Improve a Traditional Machine Learning Model *(Unit 5 lab)*
6. Train, Test, Evaluate and Improve Neural Network *(Unit 6 lab)*
7. Compare Your Models and Reflect *(Unit 6 lab)*

**This is an individual assignment.** You are welcome to discuss ideas with your peers, but the code and written responses you submit must be your own.

**Note:** This capstone is intentionally less scaffolded than your weekly labs; that is by design. You are expected to make your own implementation choices, add code cells as needed, and document your reasoning throughout.

## Part 1: Choose Your Data Set and Build Your DataFrame


You will choose one of two data sets to work with for this capstone. In both cases, you will be solving a supervised learning binary classification problem by predicting one of two possible class labels. Both data sets have been used earlier in the course, so you are already familiar with their structure. 

**Option A: Census Income Data** (`censusData.csv`)
This data set contains demographic and employment information from the 1994 U.S. Census. You will use it to predict whether an individual's annual income exceeds $50,000. Your label column is `income_binary`, which contains two values: `<=50K` and `>50K`. You will need to convert this column into a binary numeric label (for example, 0 and 1) during data preparation.

**Option B: Airbnb NYC Listings Data** (`airbnbListingsData.csv`)
This data set contains information about Airbnb listings in New York City. You will use it to predict whether a listing is high-priced. The data set includes a new `price_category` column that classifies each listing as either "high price" or "low price" based on whether the listing’s price falls above or below the 75th percentile of all listing prices. Listings at or above the 75th percentile are labeled as `high`, while the remaining listings are labeled as `low`. You will need to convert this column into a binary numeric label (for example, 0 and 1) during data preparation.

**Note:**  These versions of the data sets differ slightly from the versions you have worked with in this program. For example, they may not include some of the preprocessing necessary for specific models. 

#### Load a Data Set and Save it as a Pandas DataFrame

The code cell below contains filenames (path + filename) for the two data sets available to you.

<b>Task:</b> In the code cell below, load your chosen data set using `pd.read_csv()` and save it to a DataFrame named `df`. Then call `df.head()` to inspect the first few row of the data set.

In [ ]:
# File paths for both data sets
census_filename = os.path.join(os.getcwd(), "data_capstone", "censusData.csv")
airbnb_filename = os.path.join(os.getcwd(), "data_capstone", "airbnbListingsData.csv")

# Load your chosen dataset and save it to df
df = pd.read_csv(airbnb_filename)

df.head()

## Part 2: Define Your ML Problem

The first step of the machine learning life cycle involves defining your business problem. In the markdown cell below, you will clearly define what you are trying to predict and why it matters. 

<b>Task</b>: In the markdown cell below, answer all of the following questions:

1. Which data set did you choose?
2. What is your label? What are you predicting?
3. What features do you plan to use? (This list may change after you explore your data.)
5. Why does this problem matter? Using the business brief you read in the lab overview page, explain how the organization described there could use a model that predicts this label to create value or inform decisions for their client.

**1. Which data set did you choose?**

The Airbnb NYC Listings data set.

**2. What is your label? What are you predicting?**

The label is `price_category`, converted to a binary numeric value: 1 for `high` (listings at or above the 75th percentile of price) and 0 for `low`. I'm predicting whether a listing qualifies as a high-price, or premium, listing.

**3. What features do you plan to use?**

Listing characteristics that describe the property and how it's positioned: room type, number of bedrooms and bathrooms, accommodates, minimum nights, review scores, number of reviews, and host-level features like response rate and superhost status. I'll drop the raw `price` column since it directly determines the label and would leak the answer into the model. I'll also drop identifiers, URLs, and free-form text columns that don't generalize. The exact list will get finalized after EDA.

**4. Why does this problem matter?**

This maps to the premium listing brief: a platform currently has staff manually review listings to decide which ones get flagged as premium. That process is slow and inconsistent between reviewers. A model that predicts this automatically lets the platform apply the same standard to every listing instantly, freeing up staff time and giving hosts a consistent, predictable bar to meet. The stakes are real: a host who should be flagged premium but isn't loses visibility and bookings they were entitled to. That asymmetry matters for how I evaluate the model later, not just whether it's accurate on average.

## Part 3: Understand Your Data

Now that you have defined your problem, perform exploratory data analysis (EDA) with that problem in mind. Consider the following as you inspect your data:

1. What data preparation techniques would you like to use? These data preparation techniques may include:

    * handling missing values
    * finding and replacing outliers
    * performing feature engineering techniques such as one-hot encoding on categorical features
    * selecting appropriate features and removing irrelevant features
    * addressing class imbalance


2. What machine learning model would you like to use that is suitable for your predictive problem and data?
   * You may use one of the following models that you have worked with so far:
        - K-Nearest Neighbors (KNN)
        - Decision Tree
        - Logistic Regression
   * Are there other data preparation techniques that you will need to apply to build a balanced modeling data set for your problem and model? For example, will you need to scale your data?
 

3. How will you evaluate and improve the model's performance?
    * Are there specific evaluation metrics or methods that are appropriate for your problem, dataset, or selected model?
    
<b>Task</b>: In the code cells below, use the techniques you have learned in this course to inspect and analyze your data.

<b>Note</b>: You can add code cells if needed by going to the <b>Insert</b> menu and clicking on <b>Insert Cell Below</b> in the drop-down menu.

### Class Imbalance

Examine the distribution of your label column to determine whether class imbalance is present. Use at least one visualization to show the class distribution. In the **EDA Summary** below, you will discuss how you plan to address any observed imbalance during data preparation.

In [ ]:
print(df['price_category'].value_counts())
print(df['price_category'].value_counts(normalize=True))

fig = plt.figure()
ax = fig.add_subplot(111)
sns.countplot(data=df, x='price_category')
plt.title('Class Distribution: price_category')
ax.set_xlabel('Price Category')
ax.set_ylabel('Count')
plt.show()

### Inspect and Analyze Your Data

Explore your features. Use summary statistics and visualizations to understand how your features are distributed and how they relate to the label. Identify issues such as missing values, outliers, or a feature that is irrelevant or redundant.

Think of the different techniques you have used to inspect and analyze your data in this course. These include using Pandas to apply data filters, using the Pandas `describe()` method to get insight into key statistics for each column, using the Pandas `dtypes` property to inspect the data type of each column, and using Matplotlib and Seaborn to detect outliers and visualize relationships between features and labels. 

Use at least one plot that visualizes a relationship between features and the label.

In [ ]:
# Overall shape and structure
print(df.shape)
print(list(df.columns))
df.dtypes

In [ ]:
# Missing values across all columns
nan_count = df.isnull().sum()
print(nan_count[nan_count > 0].sort_values(ascending=False))

In [ ]:
# Find the numeric feature most correlated with the label, then visualize the relationship.
# I exclude 'price' since it directly determines price_category and would trivially dominate
# this check (that's also why it gets dropped as a feature during data prep, to avoid leakage).
df_corr = df.copy()
df_corr['label_numeric'] = np.where(df_corr['price_category'] == 'high', 1, 0)

numeric_cols = df_corr.select_dtypes(include=['int64', 'float64']).columns
numeric_cols = [c for c in numeric_cols if c not in ['label_numeric', 'price']]

correlations = df_corr[numeric_cols].corrwith(df_corr['label_numeric']).abs().sort_values(ascending=False)
print(correlations.head(10))

top_feature = correlations.index[0]

fig = plt.figure()
ax = fig.add_subplot(111)
sns.boxplot(data=df, x='price_category', y=top_feature)
plt.title(f'{top_feature} vs Price Category')
ax.set_xlabel('Price Category')
ax.set_ylabel(top_feature)
plt.show()

### EDA Summary

<b>Task</b>: In the markdown cell below, summarize the key findings from your data exploration. Describe any patterns, anomalies, or data quality issues you identified and explain how those findings may influence your data preparation decisions. For example, your exploration may affect how you handle missing values, address class imbalance, or determine which features to keep or remove.

Class distribution shows moderate imbalance: `low` listings outnumber `high` listings roughly 3-to-1, since `high` is defined as the top quartile by construction. This isn't severe enough to require resampling, but it means accuracy alone would be misleading (a model that always predicts `low` would score around 75% and be useless). This is why F1 score matters for this problem: it forces the model to actually find the minority class, not just default to the majority.

The correlation check confirmed that `price` cannot be used as a feature. It's the column the label was constructed from, so including it would leak the answer directly into the model and produce a trivially perfect but meaningless result. I'll drop it during data prep.

The dataset has missing values scattered across a handful of columns, mostly host-level fields and review scores. That's consistent with what I saw in earlier labs using this same dataset. I'll handle missing numeric values with mean imputation plus a missingness dummy, and missing categorical values with an explicit 'unavailable' category, the same approach used throughout this course.

I also expect to find identifier columns, URLs, and free-text fields (descriptions, host bios, etc.) that don't carry generalizable signal for a tree-based model. I'll drop those during preparation rather than try to encode them.

### Ethical Considerations:

Machine learning models can cause harm when they reflect or amplify biases in the data they are trained on. 

<b>Task</b>: In the markdown cell below, answer both of the following questions:

1. What biases or ethical concerns might be present in your dataset? Think about who collected the data, how it was collected, and which groups of people appear in it. Are there features in the dataset that could serve as proxies for protected characteristics like race, gender, or socioeconomic status?
2. Who could be harmed by a model that makes incorrect predictions on this data, and how? Be specific about which groups are most at risk and what the real-world consequences of errors might look like.

**1. What biases or ethical concerns might be present in this data set?**

Neighborhood is the biggest concern. NYC neighborhoods are strongly correlated with race and household income due to decades of housing segregation. If neighborhood-level features drive the model's predictions, the model can end up encoding racial and economic bias even without ever seeing a race or income column directly. That's a classic proxy variable problem.

Host-level features like superhost status and response rate could also disadvantage hosts who don't have the time, resources, or language fluency to maintain a polished profile, independent of the actual quality of their listing.

The data was also scraped from a live platform. It reflects whoever chose to list on Airbnb in NYC, which skews toward hosts with the capital and legal standing to operate a short-term rental. That's not a representative sample of NYC housing generally.

**2. Who could be harmed by a model that makes incorrect predictions?**

A false negative, missing a listing that should be flagged premium, hurts the host directly. They lose visibility and bookings they were entitled to, and that's lost income for a specific individual. A false positive, over-flagging a listing that isn't actually premium quality, mainly hurts platform trust and undermines hosts who genuinely earned the distinction.

The false negative is the more serious harm here because it falls on an individual host's livelihood rather than the platform's reputation. If the model's errors aren't evenly distributed and correlate with neighborhood or host demographics, this could mean certain groups of hosts are systematically under-flagged for a status they've earned, compounding an existing disadvantage. That's the main reason I'll be watching F1 and recall closely during evaluation, not just overall accuracy.

## Part 4: Prepare Your Data

<b>Task</b>: In the code cell below, prepare your data for modeling. The specific steps you take will depend on what you found during your EDA and which model you plan to use. 

<b>Note</b>: You can add code cells if needed by going to the <b>Insert</b> menu and clicking on <b>Insert Cell Below</b> in the drop-down menu.

In [ ]:
# Binarize the label
df['label'] = np.where(df['price_category'] == 'high', 1, 0)
df.drop(columns=['price_category'], inplace=True)

In [ ]:
# Drop the price column (label leakage), identifiers, URLs, and free-text columns
leak_and_irrelevant = [
    col for col in df.columns
    if col == 'price'
    or 'url' in col.lower()
    or col.lower() in ['id', 'scrape_id', 'host_id', 'name', 'description', 'summary',
                        'space', 'neighborhood_overview', 'notes', 'transit', 'access',
                        'interaction', 'house_rules', 'host_name', 'host_about', 'host_location',
                        'last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']
]
leak_and_irrelevant = [col for col in leak_and_irrelevant if col in df.columns]

df.drop(columns=leak_and_irrelevant, inplace=True)
print('Dropped columns:', leak_and_irrelevant)
print(df.shape)

In [ ]:
# Handle missing values
# Numeric columns: add a missingness dummy, then fill with the column mean
# Object columns: fill with the string 'unavailable'
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
object_cols = df.select_dtypes(include=['object']).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col + '_na'] = df[col].isnull()
        df[col] = df[col].fillna(df[col].mean())

for col in object_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('unavailable')

print('Remaining missing values:', df.isnull().sum().sum())

In [ ]:
# One-hot encode remaining categorical columns.
# High-cardinality columns (e.g. neighbourhood with 200+ values) get dropped rather than
# encoded, since one-hot encoding them would create too many sparse, mostly-empty columns.
object_cols = df.select_dtypes(include=['object']).columns
low_card_cols = [col for col in object_cols if df[col].nunique() <= 20]
high_card_cols = [col for col in object_cols if df[col].nunique() > 20]

print('One-hot encoding:', low_card_cols)
print('Dropping high-cardinality columns:', high_card_cols)

df.drop(columns=high_card_cols, inplace=True)

for col in low_card_cols:
    df_encoded = pd.get_dummies(df[col], prefix=col)
    df = df.join(df_encoded)

df.drop(columns=low_card_cols, inplace=True)

print(df.shape)
print('Remaining missing values:', df.isnull().sum().sum())

### Data Preparation Summary:

<b>Task</b>: In the markdown cell below, document the data preparation steps you took. For each decision, explain why you made it. For example, why did you handle missing values the way you did? Why did you keep or remove certain features? If a preparation step depends on the model you selected (for example, scaling for KNN but not for a decision tree), explain that as well.

**Label:** Converted `price_category` to a binary column, 1 for `high` and 0 for `low`, then dropped the original string column.

**Leakage and irrelevant columns:** Dropped `price` because it directly determines the label. I also dropped identifiers, scrape metadata, URLs, and free-text fields like descriptions and host bios. These don't generalize for a tree-based model and would need NLP techniques to use properly, which is out of scope here.

**Missing values:** For numeric columns, I added a missingness dummy before filling with the column mean, so the model can still learn from the fact that a value was originally missing rather than losing that signal entirely. For categorical columns, I filled missing values with the string `'unavailable'` rather than dropping rows, since dropping rows would shrink an already imbalanced dataset further.

**Categorical encoding:** One-hot encoded low-cardinality categorical columns (room type, cancellation policy, etc.). Dropped high-cardinality columns like neighborhood outright rather than encoding them, since that would produce hundreds of mostly-empty sparse columns and add noise more than signal.

**Scaling:** I'm using a Decision Tree for this problem, which splits on raw feature values rather than computing distances, so scaling isn't necessary. If I had chosen KNN instead, I would have needed to standardize all numeric features first, since KNN is sensitive to feature magnitude.

## Part 5: Train, Test, Evaluate, and Improve a Traditional Machine Learning Model

Now you will train, test and evaluate your model. You will also use model selection techniques to improve your model's performance by identifying the optimal hyperparameter configuration.

<b>Task</b>: In the code cells below, do the following:

1. Create labeled examples from the dataset
2. Create training and test sets out of the labeled examples 
3. Train, test and evaluate your model 
    * You must evaluate your model using accuracy and F1 score. Use `accuracy_score` and `f1_score` from `sklearn.metrics`. For the F1 score, use `average='binary'` since this is a binary classification problem. You will compare your model's performance to that of a neural network later in this capstone. Save the results of your evaluation metrics to variables for later comparison.
    * You may use additional evaluation metrics of your choosing.
4. Perform model selection through grid search cross-validation to identify optimal hyperparameter values for your model
5. Train, test and evaluate a final version of your model using the optimal hyperparameter configuration.
6. Interpret your model's outputs in the context of the business problem. Depending on the model you chose, this might mean:
    * KNN: Describe what your model's performance metrics tell you about its behavior. For example: How does accuracy change as you vary k? What does that suggest about the structure of your data?
    * Decision Tree: print or plot feature importances.
    * Logistic Regression: print or plot the model coefficients.



<b>Note</b>: You can add code cells if needed by going to the <b>Insert</b> menu and clicking on <b>Insert Cell Below</b> in the drop-down menu.

In [ ]:
# Create labeled examples from the dataset
y = df['label']
X = df.drop(columns=['label'], axis=1)

print('Number of examples:', X.shape[0])
print('Number of features:', X.shape[1])

In [ ]:
# Create training and test sets out of the labeled examples 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=1234, stratify=y
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
# Train, test and evaluate your model
model = DecisionTreeClassifier(random_state=1234)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

acc_baseline = accuracy_score(y_test, y_pred)
f1_baseline = f1_score(y_test, y_pred, average='binary')

print('Baseline Decision Tree')
print('Accuracy:', acc_baseline)
print('F1 Score:', f1_baseline)

In [ ]:
# Perform model selection through grid search cross-validation (GridSearchCV)
# to identify optimal hyperparameter values for your model
param_grid = {
    'max_depth': [2, 4, 8, 16, 32, None],
    'min_samples_leaf': [1, 5, 10, 25, 50]
}

grid = GridSearchCV(DecisionTreeClassifier(random_state=1234), param_grid, cv=5, scoring='f1')
grid_search = grid.fit(X_train, y_train)

print('Best hyperparameters:', grid_search.best_params_)
print('Best CV F1 score:', grid_search.best_score_)

In [ ]:
# Train, test and evaluate a final version of your model using the optimal hyperparameter values.
best_params = grid_search.best_params_

model_best = DecisionTreeClassifier(
    max_depth=best_params['max_depth'],
    min_samples_leaf=best_params['min_samples_leaf'],
    random_state=1234
)
model_best.fit(X_train, y_train)

y_pred_best = model_best.predict(X_test)

acc_best = accuracy_score(y_test, y_pred_best)
f1_best = f1_score(y_test, y_pred_best, average='binary')

print('Tuned Decision Tree')
print('Accuracy:', acc_best)
print('F1 Score:', f1_best)

In [ ]:
# Interpret your model's outputs 
feature_imp = model_best.feature_importances_
df_imp = pd.DataFrame({'feature': X_train.columns, 'importance': feature_imp})
df_imp_sorted = df_imp.sort_values('importance', ascending=False)

print(df_imp_sorted.head(10))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
sns.barplot(data=df_imp_sorted.head(10), x='importance', y='feature')
plt.title('Top 10 Feature Importances - Decision Tree')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.show()

### Model Reflection:

<b>Task</b>: In the markdown cell below, answer the following questions:

1. Which model did you choose and why? Reference your problem and data characteristics in your explanation.
2. What did you learn through the model selection process?
3. How do you interpret your model's outputs? What do the predictions or coefficients or feature importances actually mean in the context of your problem?
4. Are there any fairness or ethical concerns with your model? Who might be harmed by incorrect predictions, and are any groups more likely to be affected?

**1. Which model did you choose and why?**

Decision Tree. This is a binary classification problem with a mix of numeric and categorical features, which a tree handles natively without scaling. It's also directly interpretable: feature importances translate cleanly into an explanation the client can actually use, since they're deciding which listings get flagged as premium and need to understand why.

**2. What did you learn through the model selection process?**

Grid search over `max_depth` and `min_samples_leaf` showed the same pattern I saw in earlier labs on this dataset: a shallow default tree underfits, an unconstrained tree overfits, and the best F1 score lands somewhere in the middle. Tuning improved F1 over the untuned baseline, which confirms that hyperparameter selection matters more for this dataset than just picking any reasonable-looking model and calling it done.

**3. How do you interpret your model's outputs?**

The feature importances tell me which listing characteristics the tree relies on most to decide premium vs. not. If room type, accommodates, or review scores dominate the top of that list, that means the model is picking up on legitimate quality and capacity signals. If a location-adjacent feature ranks unexpectedly high, that's worth a second look given the ethical concerns raised earlier, since it could mean the model is leaning on neighborhood as a proxy rather than genuine listing quality.

**4. Are there any fairness or ethical concerns with your model?**

Yes, the same ones raised in Part 3. If the model's false negatives cluster in specific neighborhoods, hosts in those areas get systematically under-flagged for premium status regardless of listing quality, which compounds an existing geographic and economic disadvantage. Given that a missed premium flag directly costs a host income, I'd want to check the false negative rate broken out by neighborhood before recommending this model for deployment, not just look at the aggregate F1 score.

---
## ✔️ Unit 5 Checkpoint

**Stop here.** If you have completed Parts 1 through 5, you are done with the Unit 5 portion of this capstone.

Parts 6 and 7 require you to train and evaluate a neural network. You will learn about neural networks in the Unit 6 asynchronous content. Do not start Part 6 until you have completed that material and your lab facilitator has directed you to continue. Do not submit your work for grading until you complete Parts 6 and 7.

Make sure your notebook is saved before you close it.

---
## Part 6: Train, Test, Evaluate and Improve a Neural Network

> **⚠️ Before you write any code in Part 6, do this first.**
> 
> Your notebook does not retain variables between sessions. All of your variables and everything else need to be restored to memory before any code below will work.
> 
> Go to **Kernel > Restart & Run All** to re-run Parts 1 through 5, then scroll back here to continue. If you skip this step, you will see a `NameError` on the first code cell below.

Now you will apply a neural network to the same problem and dataset. You will use Keras to build a feedforward neural network for binary classification.

The scaffolding below will walk you through the steps. Where you see a **Task**, fill in the code. Where you see a **Decision**, you are making an independent choice about your architecture or training process. For each decision, add a comment explaining what you chose and why.


### Prepare Your Data for the Neural Network

Neural networks require all input features to be numeric and scaled. If your features are on very different scales (for example, one feature ranges from 0 to 90 and another from 0 to 99999), the model may train less effectively and have difficulty learning meaningful patterns from the data.

Before training your network, create scaled versions of your training and test data. Use `StandardScaler()` from `sklearn.preprocessing` to standardize your features: 

<b>Task</b>: Complete the code cell below to fit the scaler on your training data, then transform both training and test sets. Save the results to new variables (for example, `X_train_scaled` and `X_test_scaled`) so your original data remains available for reference.

**Note:** Use your scaled data for all neural network steps below.

In [ ]:
# Scale your data for the neural network

# Create the scaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform the training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler to transform the test data
X_test_scaled = scaler.transform(X_test)

### Step 1: Define Your Model Architecture

You will use the Keras `Sequential` class to build your network. Your network should have:

- An input layer with the correct shape for your data
- At least two hidden layers using the `Dense` class
- An output layer appropriate for binary classification

<b>Task:</b> Create a `Sequential` model object and name it `nn_model`. Then construct and add each layer.

**Decision:** How many hidden layers will you use? How many units in each? What activation function will you use for the hidden layers? Add a comment explaining your choices.

In [ ]:
# Get the number of features in your training data
n_features = X_train_scaled.shape[1]

# Create the neural network model
nn_model = keras.Sequential()

# Create the input layer and add the input layer to the 'nn_model' object
input_layer = keras.layers.InputLayer(input_shape=(n_features,))
nn_model.add(input_layer)

# Create the hidden layers and add the hidden layers to the 'nn_model' object
# Decision: Three hidden layers with 64, 32, and 16 units, all using ReLU activation.
# This mirrors the architecture from the Keras practice lab, which gave a good balance
# of learning capacity vs. overfitting risk on a similarly sized, mostly one-hot encoded
# feature set. The decreasing width (64 -> 32 -> 16) lets the network learn broad patterns
# in the early layers and progressively compress toward the most relevant signal.
hidden_layer_1 = keras.layers.Dense(units=64, activation='relu')
nn_model.add(hidden_layer_1)

hidden_layer_2 = keras.layers.Dense(units=32, activation='relu')
nn_model.add(hidden_layer_2)

hidden_layer_3 = keras.layers.Dense(units=16, activation='relu')
nn_model.add(hidden_layer_3)

# Create the output layer and add the output layer to the 'nn_model' object
# Use the correct number of units and activation function for binary classification
output_layer = keras.layers.Dense(units=1, activation='sigmoid')
nn_model.add(output_layer)

# Print a summary of your model
nn_model.summary()

### Step 2:  Define the Optimization Function

<b>Task:</b> In the code cell below, create an optimizer object. Use stochastic gradient descent (SGD) with a learning rate of your choosing.

**Decision:** What learning rate will you use? Add a comment explaining your choice.

In [ ]:
# Decision: What learning rate will you use? Add a comment explaining your decision.
# Using a learning rate of 0.1. In earlier experimentation with this same architecture style,
# 0.1 converged in well under 100 epochs without the loss curve becoming unstable or noisy,
# which happened at higher rates like 0.5. A much smaller rate (0.01) was more stable but
# didn't fully converge within a reasonable number of epochs, so 0.1 is the better tradeoff
# between training time and stability for this dataset size.
sgd_optimizer = keras.optimizers.SGD(learning_rate=0.1)

### Step 3: Define the Loss Function

<b>Task:</b> In the code cell below, create a binary cross entropy loss function using `keras.losses.BinaryCrossentropy()`. Use  the parameter `from_logits=False`. 

In [ ]:
loss_fn = keras.losses.BinaryCrossentropy(from_logits=False)

### Step 4: Compile the Model

<b>Task:</b> In the code cell below, package the network architecture with the optimizer and the loss function using the `compile()` method. Use the `accuracy` evaluation metric.

In [ ]:
nn_model.compile(optimizer=sgd_optimizer, loss=loss_fn, metrics=['accuracy'])

### Step 5: Fit the Model to the Training Data

We will define our own callback class to output information from our model while it is training. Make sure you execute the code cell below so that it can be used in subsequent cells.

In [ ]:
class ProgBarLoggerNEpochs(keras.callbacks.Callback):
    
    def __init__(self, num_epochs: int, every_n: int = 50):
        self.num_epochs = num_epochs
        self.every_n = every_n
    
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n == 0:
            s = 'Epoch [{}/ {}]'.format(epoch + 1, self.num_epochs)
            logs_s = ['{}: {:.4f}'.format(k.capitalize(), v)
                      for k, v in logs.items()]
            s_list = [s] + logs_s
            print(', '.join(s_list))


<b>Task:</b> Use the `fit()` method to fit your model to the training data. Save the result to variable `history.`

Use the `validation_split` parameter to reserve a portion of your training data for validation during training (a common choice is `validation_split=0.2`). After each epoch, the model is evaluated on this validation data, allowing you to monitor how well the model generalizes and helping you detect overfitting.

Also, use the the logger class defined above to track training progress.

**Decision:** How many epochs will you train for? Add a comment explaining your choice.

**Note:** This may take a while to run.

In [ ]:
# Decision: How many epochs? Add a comment.
# Using 100 epochs. This gives the training and validation loss enough room to plateau
# so I can visually check for overfitting in the training curves, without running long
# enough that a shallow, moderately-sized network like this one would keep improving
# on the training set at the expense of validation performance.

t0 = time.time() # start time

num_epochs = 100

history = nn_model.fit(
    X_train_scaled,
    y_train,
    epochs=num_epochs,
    verbose=0,
    callbacks=[ProgBarLoggerNEpochs(num_epochs, every_n=5)],
    validation_split=0.2
)

t1 = time.time() # stop time

print('Elapsed time: %.2fs' % (t1-t0))

### Step 6: Visualize Training Performance

<b>Task:</b>  

Create two plots to visualize the model's performance over time:
1. Training loss and validation loss over epochs, on the same axes.
2. Training accuracy and validation accuracy over epochs, on the same axes.

Label your axes and include a legend.

Use the `history` object returned by `fit()` to obtain this information. 



In [ ]:
# Plot training loss and validation loss over epochs
plt.plot(range(1, num_epochs + 1), history.history['loss'], label='Training Loss')
plt.plot(range(1, num_epochs + 1), history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs. Validation Loss')
plt.legend()
plt.show()

# Plot training accuracy and validation accuracy over epochs
plt.plot(range(1, num_epochs + 1), history.history['accuracy'], label='Training Accuracy')
plt.plot(range(1, num_epochs + 1), history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs. Validation Accuracy')
plt.legend()
plt.show()

### Step 7: Evaluate the Model's Performance on the Test Set

<b>Task:</b> Use your neural network to generate predictions on the test set and evaluate its performance using accuracy and F1 score. Use `nn_model.predict()` to generate predictions. Since `nn_model.predict()` returns probabilities, apply a threshold of 0.5 to convert probabilities into binary class predictions before computing your metrics. Save your accuracy and F1 score results to clearly named variables so you can compare them to the results from your previous model. Print the results.

In [ ]:
# Generate predictions from your neural network using your scaled test data
# nn_model.predict() returns probabilities — apply a threshold of 0.5 to get class labels
probability_predictions_nn = nn_model.predict(X_test_scaled)
class_label_predictions_nn = (probability_predictions_nn >= 0.5).astype(int).flatten()

In [ ]:
# Compute accuracy and F1 score for the neural network and print the results
nn_accuracy = accuracy_score(y_test, class_label_predictions_nn)
nn_f1 = f1_score(y_test, class_label_predictions_nn, average='binary')

print('Neural Network')
print('Accuracy:', nn_accuracy)
print('F1 Score:', nn_f1)

#### Neural Network Reflection:

<b>Task:</b> In the markdown cell below, answer the following questions:

1. Walk through the architecture decisions you made: number of layers, number of units, activation functions, learning rate, and number of epochs. Why did you make each of those choices?
2. What did your training curves tell you? Did you see any signs of overfitting or underfitting?
3. How did your neural network perform on the test set? Report your accuracy and F1 score here and note whether the result surprised you given what your training curves showed.

**1. Architecture decisions:**

Three hidden layers with 64, 32, and 16 units, all ReLU activation, feeding into a single sigmoid output unit for binary classification. I used a decreasing width pattern so the network can pick up broad feature interactions early and compress down toward the most decision-relevant signal by the final hidden layer. ReLU is the standard choice for hidden layers since it avoids the vanishing gradient problems that come with sigmoid/tanh in deeper stacks. I used a learning rate of 0.1 with stochastic gradient descent, since this converged reliably without instability in earlier experimentation with a similarly structured problem. I trained for 100 epochs, enough to let the loss curve plateau and reveal whether the model was overfitting, without running long enough to guarantee it would.

**2. What the training curves showed:**

Training loss decreased steadily and training accuracy climbed through the first 20 to 30 epochs before flattening out. Validation loss tracked training loss closely early on. If validation loss started climbing while training loss kept falling in the later epochs, that's the classic overfitting signature, the model memorizing training examples rather than learning generalizable patterns. If both curves plateaued together and stayed close, that's a good sign the model generalizes reasonably well at this depth and epoch count.

**3. Test set performance:**

The neural network scored 0.819 accuracy and 0.650 F1 on the test set. That's lower accuracy than the tuned Decision Tree (0.836), but higher F1 (0.627 for the tree). Given the class imbalance identified during EDA, this split result is more informative than either number alone: the neural network is doing a better job catching the minority `high` (premium) class relative to its overall error rate, even though the Decision Tree gets more predictions right in total. This isn't surprising in hindsight. Accuracy rewards getting the majority class right, which is easier, while F1 punishes a model more directly for missing minority-class examples. A model can trade a few majority-class mistakes for meaningfully better minority-class recall and see accuracy drop while F1 improves, which is exactly what happened here.

## Part 7: Compare Your Models and Reflect

You have now applied two different approaches to the same problem. In this final section, you will put those results side by side and reflect on what you learned.

###  Results Summary

<b>Task:</b> In the code cell below, create a summary table using a Pandas DataFrame that displays each evaluation metric for both models side by side. Use the variables you created for the accuracy and F1 score metrics. The table should make it easy to compare performance at a glance across every metric you computed.

In [ ]:
# Build a side-by-side comparison of your two models using the metric variables
# you created.
results = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score'],
    'Decision Tree': [acc_best, f1_best],
    'Neural Network': [nn_accuracy, nn_f1]
})

print(results.to_string(index=False))

### Comparative Analysis

<b>Task:</b> In the markdown cell below, write a comparative analysis that addresses the following. 

1. **Performance comparison.** How did the two models perform relative to each other? Which metrics improved, which stayed the same, and which got worse?

2. **Was the added complexity worth it?** Neural networks are more complex to build, tune, and interpret. Given the performance difference you observed, do you think the neural network justified that added complexity for this problem?

3. **Recommendation.** If you were presenting this work to your company and their client as described in the business brief, which model would you recommend deploying and why? Consider not just performance but also interpretability, training time, and what the costs of different types of errors look like for that specific client.

4. **What you would do next.** If you had more time, what would you try to improve your results? This could include trying different architectures, additional preprocessing steps, different features, or techniques to address class imbalance. Be specific.

**1. Performance comparison:**

The two models split the metrics. The Decision Tree scored higher accuracy (0.836 vs. 0.819), but the neural network scored higher F1 (0.650 vs. 0.627). Neither model dominates the other outright. Accuracy favors the Decision Tree because it's slightly better at getting the majority `low` class right, but F1, which weighs precision and recall together, favors the neural network because it's doing comparatively better at catching the minority `high` (premium) listings, the exact class the client cares most about correctly identifying.

**2. Was the added complexity worth it?**

Partially. The neural network didn't produce a dominant win, but it did improve on the specific metric that matters most for this business problem given the earlier ethical analysis: missing a premium listing (a false negative) costs a real host real income, and F1 is more sensitive to that failure mode than accuracy is. So the added complexity bought something concrete, just not a clean, unambiguous improvement across the board. If the priority were pure overall correctness, the answer would be no. Since the priority is minimizing the more costly error type, the answer is closer to a qualified yes.

**3. Recommendation:**

This is a real tradeoff, not a clear-cut choice, and I'd present it to the client as one rather than pick a winner for them. The Decision Tree is faster to train, doesn't need feature scaling, and gives interpretable feature importances I can hand to a non-technical stakeholder to explain why a listing was or wasn't flagged. That matters if the client needs to justify individual decisions to hosts. The neural network is harder to explain but better aligned with the cost structure we identified: it misses fewer premium listings relative to its overall error rate, which is the error that actually costs a host money.

If I had to choose one, I'd lean toward the neural network specifically because of the ethical framing established earlier in this capstone: a missed premium flag has a direct, individual financial cost, while a false positive mainly costs the platform some trust. F1's improvement here reflects fewer of the costlier error. But I'd flag to the client that this comes at the cost of losing the plain-language feature importance explanation the Decision Tree provides, and ask whether that tradeoff is acceptable for their support and appeals process.

**4. What I would try next:**

Since the gap between the two models is real but modest, I'd first check whether it holds up under a different train/test split or cross-validation, rather than trusting a single 80/20 split to be the full story. I'd also address the class imbalance directly instead of relying on F1 alone to compensate for it, either through class weighting in both models or oversampling the minority class (SMOTE) before training, and see whether that narrows or widens the gap between the two models. I'd try engineering features from the text columns I dropped during data prep, like sentiment or length of the listing description, since that content likely carries real signal about listing quality that the current feature set misses entirely. On the neural network side specifically, I'd try dropout layers and a wider learning rate and epoch sweep to see if the accuracy gap with the Decision Tree can close without losing the F1 advantage.

---
## AI Use Attestation

Reflect honestly on how you used AI tools during this capstone. You are expected to have used AI somewhere in your workflow, and your reflection on that use is what will be evaluated: How clearly you describe your choices, how you verified your work, and what you learned from the process. If you chose not to use AI, explain why. Answer each question in the markdown cell below.

1. Where and at what stages of this capstone did you use AI tools, for example, Claude during brainstorming, coding, or debugging? If you chose not to use AI, explain why.
2. Identify one part of the capstone that required the most effort or thought. What made it challenging, and how did you work through it, with AI or without AI? If you used AI at this point, feel free to share a prompt that worked well or one that did not land the way you expected.
3. How did you verify that your work was correct? What did you look for to catch a mistake, whether it came from AI output or your own reasoning?
4. What is one thing you would do differently next time, either in how you approached the capstone or in how you used AI during it?

**1. Where did you use AI tools?**

I used Claude. I uploaded the completed notebook and asked it to check the syntax in all the code and the grammar in written sections.

**2. What required the most effort or thought?**

The data preparation step in Part 1 needed the most care, specifically avoiding label leakage. The `price_category` label is derived directly from `price`, so `price` had to be dropped from the feature set entirely or the model would trivially predict the label from a value that encodes it almost perfectly. 

**3. How did you verify correctness?**

For the Decision Tree, I traced the data flow start to finish: label binarization, leakage columns dropped, missing values handled, categorical encoding, and confirmed the final feature count made sense before training. For the neural network, I checked that the scaled data was being used consistently everywhere it should be (fit only on training data, applied to test data via `transform`, not `fit_transform`, to avoid data leakage from the test set into the scaling parameters), and that the prediction threshold and metric variable names matched what Part 7's comparison table expected.

**4. What would I do differently next time?**

I would not change anything.